In [2]:
import ollama
import requests
import json
from ollama import chat
from ollama import ChatResponse

In [3]:
model = "llama3.2:1b"

#### Task 1: Interact with deployed LLM via python 


**Objective:**

Explore different techniques to interact with the deployed LLM.

**Task Description:**

1. Use Request libaray (HTTP Client) and send a POST request to interact with the LLM: [How To](https://requests.readthedocs.io/en/latest/user/quickstart/#make-a-request)

In [4]:
# Simple HTTP Request via requests

# Define the URL of the deployed LLM ( this port is forwarded from the docker container to the host system)
url = "http://localhost:11434/api/generate"

# Define the prompt as json
body_json = {
    "model": model,
    "prompt": "Describe Generative AI in two sentences."
}

# ADD HERE YOUR CODE
# HINT: Send the POST request using the json body
response = requests.post(url, json=body_json)

# Check if the request was successful
if response.status_code == 200:
    # Process the response
    response_text = response.text

    # Convert each line to json
    response_lines = response_text.splitlines()
    response_json = [json.loads(line) for line in response_lines]
    for line in response_json:
        # Print the response. No line break
        print(line["response"], end="")
else:
    print("Error:", response.status_code)


Generative AI refers to a type of artificial intelligence that can create new content, such as text, images, or music, based on patterns and algorithms, allowing for the generation of diverse and unique outputs without being explicitly programmed. This technology uses machine learning and natural language processing techniques to learn from existing data and produce novel outputs that can be used for a wide range of applications, including creative content creation, art, and simulation.

**Task Description:**

2. Use Ollama python library to interact with the LLM: [How To](https://pypi.org/project/ollama/)

- First use method ``ollama.chat(...)``
- First use method ``ollama.chat(...)`` with ``stream=True``

In [ ]:
# API Call via ollama

# ADD HERE YOUR CODE

response: ChatResponse = chat(
    model = model, 
    messages = [
        {
            'role': 'user', 'content': 'Why is the sky blue?',
        }
    ]
)


print(response["message"]["content"])

The sky appears blue to us because of a phenomenon called Rayleigh scattering, named after the British physicist Lord Rayleigh, who first explained it in the late 19th century.

Here's what happens: When sunlight enters Earth's atmosphere, it encounters tiny molecules of gases such as nitrogen and oxygen. These molecules scatter the light in all directions, but they scatter shorter (blue) wavelengths more than longer (red) wavelengths. This is because the smaller molecules are more effective at scattering the shorter wavelengths.

As a result, the blue light is distributed throughout the atmosphere, giving the sky its blue color. The amount of scattering that occurs depends on the altitude and density of the atmosphere, but in general, the sky appears blue when the sun is high in the sky and the atmosphere is relatively thin.

It's worth noting that the sky can appear more red or purple in certain conditions, such as when the sun is low in the sky or during sunrise and sunset. This is 

In [10]:
# Streaming API Call via ollama

# Response streaming can be enabled by setting stream=True, 
# modifying function calls to return a Python generator where each part is an object in the stream.

# ADD HERE YOUR CODE
stream = chat(
  model = model, messages = [{
    'role': 'user', "content": "What is VfB Stuttgart?"
  }],
  stream = True
)


for chunk in stream:
  print(chunk["message"]["content"], end="", flush=True)

VfB Stuttgart, commonly referred to as VfB Stuttgart or simply Stuttgart, is a professional football club based in Stuttgart, Germany. The club was founded in 1894 and is one of the most successful clubs in German football history.

VfB Stuttgart is currently competing in the Bundesliga, the top tier of German football, and has won numerous domestic and international titles, including three German championships, four DFB-Pokal trophies, and one UEFA Cup Winners' Cup. The club has also had significant success in European competitions, including a Champions League semifinal appearance in 1995.

The team is nicknamed "Die Roten Könige" (The Red Kings) due to its red and gold color scheme, and its home stadium is the Neckar-Auen-Stadion (Stuttgart Stadium), which has a seating capacity of over 55,000. VfB Stuttgart has a strong rivalry with neighboring club Bayern Munich, with whom they contest the traditional "Allianz Cup" competition between the two teams.

#### Task 2: Experimenting with Prompt Techniques

**Objective:**

Objective: Explore different prompt techniques (Zero Shot, One Shot, and Few Shot) by sending different types of prompts to the LLM.

![image](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*QSpK--jqPiUU_OHuZvtUWA.png)

**Task Description:**

1. Create three prompts for a sentiment analysis task: a Zero Shot prompt, a One Shot prompt, and a Few Shot prompt. Use the examples from the table above.
2. Send these prompts to the LLM and observe the differences in the responses.
3. Compare and discuss the responses.

In [9]:
# ADD HERE YOUR PROMPTS

zero_shot_prompt = """Classify this Review. Answer with one word.: I loved this movie! Sentiment: """

one_shot_prompt = """Classify this review: I loved this movie! Sentiment: Positive. 
Classify this review. Answer with one word.: 

I don't like this chair. Sentiment: """

few_shot_prompt = """ Classify this review: I loved this movie! Sentiment: Positive. 
 Classify this review: I don't like this chair. Sentiment: Negative. 
 Classify this review. Answer with one word: who would use this product? Sentiment: """

# Stream the responses and print them
for idx, prompt in enumerate([zero_shot_prompt, one_shot_prompt, few_shot_prompt]):
    prompt_type = ["Zero-Shot", "One-Shot", "Few-Shot"][idx]
    print(f"\n--- {prompt_type} Prompt ---\n")
    print(f"User Prompt:\n{prompt}\n")
    
    stream = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    
    print("Model Output:")
    for chunk in stream:
        print(chunk["message"]["content"], end="", flush=True)
    print("\n-----------------------------\n")



--- Zero-Shot Prompt ---

User Prompt:
Classify this Review. Answer with one word.: I loved this movie! Sentiment: 

Model Output:
Positive
-----------------------------


--- One-Shot Prompt ---

User Prompt:
Classify this review: I loved this movie! Sentiment: Positive. 
Classify this review. Answer with one word.: 

I don't like this chair. Sentiment: 

Model Output:
Positive
-----------------------------


--- Few-Shot Prompt ---

User Prompt:
 Classify this review: I loved this movie! Sentiment: Positive. 
 Classify this review: I don't like this chair. Sentiment: Negative. 
 Classify this review. Answer with one word: who would use this product? Sentiment: 

Model Output:
Here are the classifications:

1. I loved this movie! Sentiment: Positive.
Answer: Audience

2. I don't like this chair. Sentiment: Negative.
Answer: Consumer

3. Answer: None (this review does not provide enough information to determine who the product is for)
-----------------------------



#### Task 3: Prompt Refinement and Optimization

**Objective:** 

Refine a prompt to improve the clarity and quality of the LLM's response.

**Task Description:**

- Start with a basic prompt asking the LLM to summarize a paragraph.
- Refine the prompt by adding specific instructions to improve the summary's quality. (Example: define how long the summary should be, define on which to focus in the summary)

In [10]:
# Original prompt
original_prompt = "Summarize the following paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries."

# ADD HERE YOUR PROMPT
refined_prompt = "Summarize the following paragraph in one short sentence: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries."

# Stream the responses and print them
for idx, prompt in enumerate([original_prompt, refined_prompt]):
    prompt_type = ["Original Prompt", "Refined Prompt"][idx]
    print(f"\n--- {prompt_type} ---\n")
    print(f"User Prompt:\n{prompt}\n")
    
    stream = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    
    print("Model Output:")
    for chunk in stream:
        print(chunk["message"]["content"], end="", flush=True)
    print("\n-----------------------------\n")



--- Original Prompt ---

User Prompt:
Summarize the following paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries.

Model Output:
Here's a summary of the paragraph:

Generative AI is an emerging field that uses patterns learned from data to create new content, such as text, images, and music. This technology has various applications in creative industries and is increasingly being used to generate new ideas and content.
-----------------------------


--- Refined Prompt ---

User Prompt:
Summarize the following paragraph in one short sentence: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries.

Mode

#### [Optional] Task 4: Structured Prompting with Roles (Pirate Theme)

**Objective:**

Learn how to use structured prompts that combine role assignment, clear instructions, and examples to improve the output of language models. In this task, you will guide the AI to respond as a pirate who is also an expert in machine learning.

**Instructions:**

- Role Assignment: In your prompt, specify the role of the AI as a Machine Learning Expert who speaks like a pirate.

- Instruction: Clearly state what you want the AI to explain or discuss in pirate language.

- Examples: Provide examples to guide the AI in using pirate lingo while explaining technical concepts.

In [ ]:
# Combined Techniques Prompt with Pirate Theme

structured_prompt = """
"""

# Stream the response and print it
print("=== User Prompt ===")
print(structured_prompt)

stream = ollama.chat(
    model=model,
    messages=[{"role": "user", "content": structured_prompt}],
    stream=True,
)

print("\n=== Model Output ===")
for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)
print("\n")
